In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)

In [ ]:
csv_path = Path(r"C:\Users\user\Desktop\div\data\files\housing.csv")
df = pd.read_csv(csv_path)
df.head(3)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY


In [ ]:
outliers1 = df['median_house_value'] == df['median_house_value'].max()
outliers2 = df['housing_median_age'] == df['housing_median_age'].max()
outliers = outliers1 | outliers2

df = df.loc[~outliers, :].copy()
print(df.shape)
df.head(3)

(18572, 10)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
8,-122.26,37.84,42.0,2555.0,665.0,1206.0,595.0,2.0804,226700.0,NEAR BAY


In [ ]:
X = df.drop('median_house_value', axis='columns')
y = df['median_house_value']

In [ ]:
X['income_cat'] = pd.cut(
    X['median_income'],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
    )

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=X['income_cat'],
    test_size=0.2
    )

X_train.shape, X_test.shape

((14857, 10), (3715, 10))

In [ ]:
X_train.drop('income_cat', axis='columns', inplace=True)
X_test.drop('income_cat', axis='columns', inplace=True)

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.cluster import KMeans

class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10):
        self.n_clusters = n_clusters

    def fit(self, X, y=None):
        self.kmeans_ = KMeans(self.n_clusters)
        self.kmeans_.fit(X)
        return self  

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=1).round(3)

    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

cat_cols = ['ocean_proximity']
log_cols = [
    'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income'
    ]

cat_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])
log_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('log', FunctionTransformer(np.log1p, inverse_func=np.expm1, feature_names_out='one-to-one')),
    ('scl', StandardScaler())
])

def ratio_func(arr_2d):
    return arr_2d[:, [0]] / arr_2d[:, [1]]
def ratio_feature_names(*args, **kwargs):
    return ['ratio']

rat_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('newcol', FunctionTransformer(ratio_func, feature_names_out=ratio_feature_names)),
    ('scl', StandardScaler())
])
cluster_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('cluster', ClusterSimilarity(n_clusters=10))
])
num_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('scl', StandardScaler())
])

preprocessing = ColumnTransformer(
    transformers=[
        ('CAT', cat_pipeline, cat_cols),
        ('LOG', log_pipeline, log_cols),
        ('RAT_b/r', rat_pipeline, ['total_bedrooms', 'total_rooms']),
        ('RAT_r/h', rat_pipeline, ['total_rooms', 'households']),
        ('RAT_p/h', rat_pipeline, ['population', 'households']),
        ("GEO", cluster_pipeline, ["latitude", "longitude"]),
    ], remainder=num_pipeline
)

X_train_arr = preprocessing.fit_transform(X_train)
X_train_prepared = pd.DataFrame(X_train_arr, columns=preprocessing.get_feature_names_out())

X_test_arr = preprocessing.fit_transform(X_test)
X_test_prepared = pd.DataFrame(X_test_arr, columns=preprocessing.get_feature_names_out())

X_train_prepared

,CAT__ocean_proximity_<1H OCEAN,CAT__ocean_proximity_INLAND,CAT__ocean_proximity_ISLAND,CAT__ocean_proximity_NEAR BAY,CAT__ocean_proximity_NEAR OCEAN,LOG__total_rooms,LOG__total_bedrooms,LOG__population,LOG__households,LOG__median_income,RAT_b/r__ratio,RAT_r/h__ratio,RAT_p/h__ratio,GEO__Cluster 0 similarity,GEO__Cluster 1 similarity,GEO__Cluster 2 similarity,GEO__Cluster 3 similarity,GEO__Cluster 4 similarity,GEO__Cluster 5 similarity,GEO__Cluster 6 similarity,GEO__Cluster 7 similarity,GEO__Cluster 8 similarity,GEO__Cluster 9 similarity,remainder__housing_median_age
0,1.0,0.0,0.0,0.0,0.0,0.827720,0.231410,0.316217,0.393411,1.320583,-1.377924,0.756014,-0.033225,0.000,0.196,0.000,0.000,0.000,0.050,0.000,0.988,0.000,0.163,-0.086119
1,0.0,1.0,0.0,0.0,0.0,0.055021,0.598606,0.029953,0.495461,-0.873844,1.558783,-0.685595,-0.092348,0.009,0.000,0.640,0.493,0.405,0.000,0.000,0.000,0.000,0.000,-0.958707
2,1.0,0.0,0.0,0.0,0.0,-0.056766,-0.686316,-0.470774,-0.563708,1.804318,-1.382460,0.862961,-0.003962,0.000,0.271,0.000,0.000,0.000,0.076,0.000,0.997,0.000,0.166,-0.958707
3,1.0,0.0,0.0,0.0,0.0,-0.869925,-0.441389,0.261025,-0.412945,-0.516757,1.261610,-0.733695,0.139035,0.000,0.667,0.000,0.000,0.000,0.998,0.000,0.080,0.053,0.004,1.920833
4,0.0,0.0,0.0,1.0,0.0,-0.070307,0.016782,-0.098885,0.005817,-0.017324,0.125545,-0.214177,-0.038913,0.000,0.000,0.619,0.128,0.954,0.000,0.000,0.000,0.000,0.000,1.397280
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14852,0.0,0.0,0.0,0.0,1.0,0.051295,-0.012109,-0.367159,0.049478,0.460658,-0.268050,-0.089812,-0.086619,0.000,0.229,0.000,0.000,0.000,0.077,0.000,0.977,0.000,0.096,1.484539
14853,1.0,0.0,0.0,0.0,0.0,-1.066807,-1.360700,-1.154138,-1.262831,0.419303,-0.702200,0.185163,-0.002485,0.000,0.627,0.000,0.000,0.000,0.989,0.000,0.090,0.051,0.003,0.786468
14854,0.0,1.0,0.0,0.0,0.0,-0.451374,-0.812533,-0.641607,-0.715513,0.894401,-0.879934,0.338935,-0.008049,0.001,0.000,0.167,0.974,0.193,0.000,0.002,0.000,0.000,0.000,-1.045966
14855,0.0,1.0,0.0,0.0,0.0,1.183609,1.302559,0.797380,1.192528,-0.817543,0.091166,-0.055280,-0.081405,0.000,0.000,0.043,0.746,0.151,0.000,0.028,0.000,0.000,0.000,-0.958707


In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("linreg", LinearRegression())
    ])
lin_reg.fit(X_train, y_train)
y_pred = lin_reg.predict(X_test)

In [ ]:
y_pred

array([ 90603.4117617 , 125653.97118557, 252620.21470775, ...,
       274796.44052157, 181197.99202705, 230997.94837707], shape=(3715,))

In [ ]:
y_test.to_numpy()

array([129200.,  96300., 204700., ..., 361000., 183300., 280000.],
      shape=(3715,))

## k‑Nearest Neighbors (KNN)

KNN is conceptually very simple:

- Training:
  - Store the training data.
- Prediction for a new point $x$:
  - Find the $k$ closest training points to $x$ according to a distance metric (usually Euclidean).
  - Classification: predict the majority class among neighbors.
  - Regression: predict the average (or weighted average) of neighbors’ target values.

This is a **non‑parametric**, **instance‑based** method:
- No explicit parameter vector $\theta$, no optimization of a global loss.
- Complexity grows with the size of the dataset, especially at prediction time.

Feature scaling is important, since distances in different dimensions are combined.

### KNN Classification

Let $\mathcal{N}_k(x)$ be the set of indices of the $k$ nearest neighbors of $x$. For classification:
- Uniform weighting:
  $$
  \hat{y}(x) = \text{mode}\{y_i : i \in \mathcal{N}_k(x)\}
  $$
- Distance weighting:
  $$
  \hat{y}(x) = \text{weighted mode with weights } w_i = \frac{1}{d(x, x_i)}
  $$
  where $d$ is the distance metric. The class with the highest sum of weights wins.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier


knn_clf = Pipeline([
    ("cleaning", preprocessing), 
    ("knn", KNeighborsClassifier(n_neighbors=5, weights="uniform"))
    ])
knn_clf.fit(X_train, y_train.ravel())
y_pred = knn_clf.predict(X_test)
y_pred

In [ ]:
y_test.to_numpy()

### KNN Regression

For regression, prediction is typically the mean (or weighted mean) of neighbor targets:
$$
\hat{y}(x) = \frac{1}{k} \sum_{i \in \mathcal{N}_k(x)} y_i
$$
or
$$
\hat{y}(x) = \frac{\sum_{i \in \mathcal{N}_k(x)} w_i y_i}{\sum_{i \in \mathcal{N}_k(x)} w_i}.
$$

Key hyperparameters:
- `n_neighbors` ($k$): small values produce very flexible decision boundaries, large values produce smoother ones.
- `weights`: `"uniform"` or `"distance"`.
- `metric`: distance function (default is Euclidean).

KNN works well in low‑dimensional spaces with relatively smooth decision boundaries and not too many training samples.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

knn_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("knn", KNeighborsRegressor(n_neighbors=5, weights="distance"))
])

knn_reg.fit(X_train, y_train.ravel())
y_pred = knn_reg.predict(X_test)
y_pred

In [ ]:
y_test.to_numpy()

## Decision Trees

Decision trees form predictions by recursively splitting the input space into rectangular regions and assigning a constant prediction to each region.

At each node:
- Choose a feature $j$ and threshold $t$.
- Split data into two child nodes:
  - Left: $x_j \le t$
  - Right: $x_j > t$
- Choose $j$ and $t$ to maximize the reduction in impurity (classification) or variance (regression).

### Classification Trees

Let a leaf node contain a set of indices \(S\). The empirical class probabilities are:
$$
p_k = \frac{1}{\lvert S \rvert} \sum_{i \in S} \mathbb{1}(y_i = k).
$$

Common impurity measures:

- **Gini impurity**:
  $$
  G(S) = \sum_{k} p_k (1 - p_k) = 1 - \sum_k p_k^2.
  $$

- **Entropy**:
  $$
  H(S) = - \sum_k p_k \log_2 p_k.
  $$

When evaluating a split into left and right sets $S_L$ and $S_R$, we compute weighted impurity:
$$
I_{\text{split}} = \frac{\lvert S_L \rvert}{\lvert S \rvert} I(S_L) +
\frac{\lvert S_R \rvert}{\lvert S \rvert} I(S_R).
$$

The algorithm picks the split that minimizes $I_{\text{split}}$ (or equivalently maximizes impurity reduction).

| Day | Outlook   | Temperature | Humidity | Wind   | Play Tennis |
|-----|-----------|-------------|----------|--------|-------------|
| D1  | Sunny     | Hot         | High     | Weak   | No          |
| D2  | Sunny     | Hot         | High     | Strong | No          |
| D3  | Overcast  | Hot         | High     | Weak   | Yes         |
| D4  | Rain      | Mild        | High     | Weak   | Yes         |
| D5  | Rain      | Cool        | Normal   | Weak   | Yes         |
| D6  | Rain      | Cool        | Normal   | Strong | No          |
| D7  | Overcast  | Cool        | Normal   | Strong | Yes         |
| D8  | Sunny     | Mild        | High     | Weak   | No          |
| D9  | Sunny     | Cool        | Normal   | Weak   | Yes         |
| D10 | Rain      | Mild        | Normal   | Weak   | Yes         |
| D11 | Sunny     | Mild        | Normal   | Strong | Yes         |
| D12 | Overcast  | Mild        | High     | Strong | Yes         |
| D13 | Overcast  | Hot         | Normal   | Weak   | Yes         |
| D14 | Rain      | Mild        | High     | Strong | No          |

- Total samples: 14
- Classes: Yes (9), No (5) 

$$P(\text{YES}) = 9/14 \approx 0.643$$
$$P(\text{NO}) = 5/14 \approx 0.357$$

1. Calculate the initial Gini Impurity (Root Node):
$$G_{\text{initial}} = 1 - (P(\text{YES})^2 + P(\text{NO})^2) = 1 - (0.643^2 + 0.357^2) = 0.460$$

2. Calculate the Gini Impurity after a potential split on the feature "Outlook":
- First Node (Outlook = Sunny):
  - Total samples: 5 
  - Classes: Yes (2), No (3)
  - $P(\text{YES}) = 2/5 \approx 0.400$
  - $P(\text{NO}) = 3/5 \approx 0.600$
  - $G_{\text{Sunny}} = 1 - (P(\text{YES})^2 + P(\text{NO})^2) = 1 - (0.400^2 + 0.600^2) = 0.480$

- Second Node (Outlook = Overcast):
  - Total samples: 4
  - Classes: Yes (4), No (0)
  - $P(\text{YES}) = 4/4 = 1.000$
  - $P(\text{NO}) = 0/4 = 0.000$
  - $G_{\text{Overcast}} = 1 - (P(\text{YES})^2 + P(\text{NO})^2) = 1 - (1.000^2 + 0.000^2) = 0.000$ (Pure Node)

- Third Node (Outlook = Rain):
  - Total samples: 5
  - Classes: Yes (3), No (2)
  - $P(\text{YES}) = 3/5 \approx 0.600$
  - $P(\text{NO}) = 2/5 \approx 0.400$
  - $G_{\text{Rain}} = 1 - (P(\text{YES})^2 + P(\text{NO})^2) = 1 - (0.600^2 + 0.400^2) = 0.480$

3. Calculate the weighted Gini Impurity for the split:
  - $G_{\text{split}} = \frac{5}{14} \times G_{\text{Sunny}} + \frac{4}{14} \times G_{\text{Overcast}} + \frac{5}{14} \times G_{\text{Rain}} = \frac{5}{14} \times 0.480 + \frac{4}{14} \times 0.000 + \frac{5}{14} \times 0.480 = 0.342$

4. Determine the best split:
  - $\text{Reduction} = G_{\text{initial}} - G_{\text{split}} = 0.460 - 0.342 = 0.118$

5. This process is repeated for all other features (e.g., Temperature, Humidity, Wind), and the feature with the highest reduction in impurity (or lowest final weighted Gini score) is chosen as the root node of the decision tree. In this case, Outlook is likely a good initial split. The process then recursively continues on the resulting child nodes until all nodes are pure or other stopping criteria are met. 

NOTE: For continuous features (like Age or Income), the algorithm doesn't guess—it tests every possible midpoint between the sorted values in your data. If you have 1,000 unique values for a feature, the algorithm will test 999 split points. For massive datasets, it sometimes uses "histograms" (binning the numbers) to speed things up, but the logic remains the same: test the gaps between the numbers.

"Early stopping" criteria (pre-pruning) to prevent the tree from growing indefinitely and overfitting.
1. Natural Stopping Points

* **Pure Node:** All samples in the node belong to the same class (Gini Impurity = 0).
* **Exhausted Features:** There are no more features left to split on, or all samples have identical feature values. 

2. User-Defined Hyperparameters
These are the most common ways to control tree size manually:

* **max_depth:** The absolute maximum levels the tree can grow. If None, it expands until nodes are pure or contain fewer than min_samples_split samples.
* **min_samples_split:** The minimum number of samples required to even attempt a split. The default is 2.
* **min_samples_leaf:** The minimum number of samples required to be in a leaf node. A split will only be allowed if both resulting branches have at least this many samples.
* **max_leaf_nodes:** Limits the total number of leaf nodes in the tree. The tree grows in a "best-first" fashion until this limit is hit.
* **min_impurity_decrease:** A node will only split if the resulting reduction in impurity (Gini or Entropy) is greater than or equal to this threshold.
* **ccp_alpha:** Used for Post-Pruning. The tree grows fully first, then the algorithm trims branches that don't improve the "cost-complexity" beyond this alpha value. 

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_clf = DecisionTreeClassifier(
    criterion="gini",
    max_depth=None,
    min_samples_split=2,
    random_state=42
)

tree_clf.fit(X_train, y_train)
y_pred = tree_clf.predict(X_test)

### Regression Trees

For regression, each leaf predicts the mean of the training targets in that leaf:
$$
\hat{y}(S) = \frac{1}{\lvert S \rvert} \sum_{i \in S} y_i.
$$

Impurity is measured by variance or MSE:
$$
I(S) = \frac{1}{\lvert S \rvert} \sum_{i \in S} (y_i - \hat{y}(S))^2.
$$

The algorithm searches for splits that reduce this impurity.

### Overfitting and Regularization

Unconstrained trees tend to overfit:
- They can create very deep, complex trees that fit training data almost perfectly.

Regularization is controlled by hyperparameters such as:
- `max_depth`: maximum depth of the tree
- `min_samples_split`: minimum samples required to split a node
- `min_samples_leaf`: minimum samples required in a leaf
- `max_leaf_nodes`: maximum number of leaves

Ensemble models such as Random Forests and Gradient Boosted Trees build on top of decision trees and will be treated separately.


In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = DecisionTreeRegressor(
    max_depth=None,
    min_samples_split=2,
    random_state=42
)

tree_reg.fit(X_train, y_train)
y_pred = tree_reg.predict(X_test)

## Support Vector Classification (SVC)

### Intuition

For binary classification with labels $y_i \in \{-1, +1\}$:
- Logistic regression tries to find parameters $\theta$ that minimize the logistic loss over all points.
- SVC tries to find a hyperplane that:
  - Correctly separates the classes (if possible)
  - Maximizes the **margin**: the distance from the hyperplane to the closest training points.

Only the training points that lie exactly on or inside the margin are critical for the solution. These are the **support vectors**. Points far away from the decision boundary do not affect the final classifier once the margin is established.

### Hard‑Margin SVM (separable case)

Assume the data is linearly separable. We seek $w \in \mathbb{R}^n$, $b \in \mathbb{R}$ such that:

- Decision function: $f(x) = w^\top x + b$
- Prediction: $\hat{y} = \text{sign}(f(x))$

Constraints for perfect separation:
$
 y_i (w^\top x_i + b) \ge 1 \quad \text{for all } i
$

Margin (distance from hyperplane to closest point) is proportional to $1 / \lVert w \rVert$. So we maximize the margin by minimizing $\lVert w \rVert^2$:

$
\min_{w, b} \frac{1}{2} \lVert w \rVert^2
\quad \text{subject to} \quad
y_i (w^\top x_i + b) \ge 1.
$

### Soft‑Margin SVM (non‑separable case)

Real data is rarely perfectly separable. Introduce slack variables $\xi_i \ge 0$ that allow some violations:

$
\begin{aligned}
&\min_{w, b, \xi} && \frac{1}{2} \lVert w \rVert^2 + C \sum_{i=1}^m \xi_i \\
&\text{subject to} && y_i (w^\top x_i + b) \ge 1 - \xi_i, \quad \xi_i \ge 0.
\end{aligned}
$

Here:
- $C > 0$ is a hyperparameter:
  - Large $C$: penalizes misclassifications strongly, narrower margin, lower bias, higher variance.
  - Small $C$: allows more violations, wider margin, higher bias, lower variance.

In practice, scikit‑learn implements this using the **hinge loss**:
$$
\ell_{\text{hinge}}(y, f(x)) = \max(0, 1 - y f(x)).
$$

### Kernel Trick

The constraint uses only inner products of the form $x_i^\top x_j$. We can replace this with a kernel:
$$
K(x_i, x_j) = \phi(x_i)^\top \phi(x_j)
$$
for some (possibly infinite‑dimensional) feature map $\phi$.

Examples:
- Linear kernel: $K(x, z) = x^\top z$
- Polynomial kernel: $K(x, z) = (\gamma x^\top z + r)^d$
- RBF kernel: $K(x, z) = \exp(-\gamma \lVert x - z \rVert^2)$

This is conceptually similar to your earlier use of `PolynomialFeatures`, but the feature space can be much higher‑dimensional and is handled implicitly.

### Basic SVC Usage

Use SVC when:
- Data is not linearly separable in original space, but might be separable after a non‑linear transformation.
- Number of features is not extremely large relative to the number of samples.

Feature scaling matters (especially with RBF and polynomial kernels).

Key hyperparameters:
- `kernel`: `"linear"`, `"poly"`, `"rbf"`, `"sigmoid"`
- `C`: regularization strength
- `gamma`: controls the width of the RBF or polynomial kernel




In [ ]:
from sklearn.svm import SVC

clf = Pipeline([
    ("cleaning", preprocessing), 
    ("svc", SVC(kernel="rbf", C=1.0, gamma="scale"))
])

clf.fit(X_train, y_train.ravel())
y_pred = clf.predict(X_test)

## Support Vector Regression (SVR)

SVR applies the SVM idea to regression. Instead of class labels, we have real‑valued targets $y_i \in \mathbb{R}$.

### Intuition

Rather than minimizing squared error everywhere, SVR tries to fit a function $f(x) = w^\top x + b$ such that:
- Predictions lie within an $\varepsilon$‑wide tube around the targets whenever possible.
- Deviations larger than $\varepsilon$ are penalized by slack variables.

Thus only points **outside** the $\varepsilon$‑tube become support vectors.

### Objective

Using the $\varepsilon$‑insensitive loss:
$$
\lvert y - f(x) \rvert_\varepsilon =
\max(0, \lvert y - f(x) \rvert - \varepsilon)
$$

The optimization problem (in primal form) is:
$$
\begin{aligned}
&\min_{w, b, \xi, \xi^*} &&
\frac{1}{2} \lVert w \rVert^2 + C \sum_{i=1}^m (\xi_i + \xi_i^*) \\
&\text{subject to} &&
y_i - w^\top x_i - b \le \varepsilon + \xi_i \\
&&& w^\top x_i + b - y_i \le \varepsilon + \xi_i^* \\
&&& \xi_i, \xi_i^* \ge 0.
\end{aligned}
$$

Again, we can use kernels to obtain non‑linear regressors.

### Basic SVR Usage

Key hyperparameters:
- `kernel`, `C`, `gamma` as in SVC
- `epsilon`: width of the insensitive tube

SVR is useful for smooth, non‑linear regression when the number of features is moderate and precise control of errors is desired.

In [ ]:
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

reg = Pipeline([
    ("cleaning", preprocessing), 
    ("svr", SVR(kernel="rbf", C=1.0, epsilon=0.1, gamma="scale"))
])

reg.fit(X_train, y_train.ravel())
y_pred = reg.predict(X_test)

- Linear and logistic regression:
  - Global linear decision boundary (after any manual feature engineering).
  - Simple convex optimization, strong interpretability of coefficients.

- SVMs (SVC, SVR):
  - Still linear in a feature space, but the feature space can be extremely rich via kernels.
  - Emphasize margin and hinge‑like loss instead of squared or logistic loss.
  - Only support vectors matter for the final solution.

- KNN:
  - No explicit parameter vector, no gradient descent, no cost function in the usual sense.
  - Prediction is local and distance‑based; decision boundary can be highly irregular.

- Decision Trees:
  - Construct piecewise constant approximations of decision boundaries.
  - No gradient computations; training is greedy, based on impurity reduction.